## 1. Setup and Data Loading
This section handles all necessary imports, defines data paths, loads the image data, and splits it into training and testing sets. It also defines the custom `DefectDataset` and sets up `DataLoader` instances for efficient batching during training and evaluation.

In [ ]:
import os
import random
import cv2
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
from tqdm import tqdm

ok_path = "*"
def_path = "*"

data = []

for f in os.listdir(ok_path):
    data.append((os.path.join(ok_path, f), 0))

for f in os.listdir(def_path):
    data.append((os.path.join(def_path, f), 1))

random.shuffle(data)

ok_data = [x for x in data if x[1] == 0]
def_data = [x for x in data if x[1] == 1]

ok_train, ok_test = train_test_split(ok_data, test_size=0.5, random_state=42)
def_train, def_test = train_test_split(def_data, test_size=0.5, random_state=42)

train_data = ok_train + def_train
test_data = ok_test + def_test

random.shuffle(train_data)
random.shuffle(test_data)

class DefectDataset(Dataset):
    def __init__(self, data, transform=None):
        self.data = data
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_path, label = self.data[idx]

        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)

        if self.transform:
            img = self.transform(image=img)["image"]

        return img, torch.tensor(label, dtype=torch.float32)


## 2. Data Augmentation and DataLoaders
Here, we define the image transformations and augmentations for training and testing. These help improve model generalization. The `DataLoader`s are then initialized to manage data loading in batches.

In [ ]:
train_tfms = A.Compose([
    A.Resize(224, 224),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.2),
    A.Rotate(limit=15, p=0.5),
    A.RandomBrightnessContrast(p=0.5),
    A.GaussNoise(p=0.3),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=10, p=0.5),
    A.CoarseDropout(max_holes=8, max_height=16, max_width=16, p=0.3),
    A.Normalize(),
    ToTensorV2()
])

test_tfms = A.Compose([
    A.Resize(224, 224),
    A.Normalize(),
    ToTensorV2()
])

train_ds = DefectDataset(train_data, transform=train_tfms)
test_ds = DefectDataset(test_data, transform=test_tfms)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=2)


## 3. Model Definition and Loss Function
This section defines the neural network architecture using `timm` for the backbone and a linear layer for classification. It also implements a custom `FocalLoss` function, which is often used for imbalanced datasets.

In [ ]:
class Model(nn.Module):
    def __init__(self):
        super().__init__()

        self.backbone = timm.create_model(
            "tf_efficientnet_b0.ns_jft_in1k",
            pretrained=True,
            num_classes=0
        )

        self.classifier = nn.Linear(1280, 1)

    def forward(self, x):
        x = self.backbone.forward_features(x)
        x = x.mean(dim=[2,3])
        return self.classifier(x)

class FocalLoss(nn.Module):
    def __init__(self, gamma=2):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
        self.gamma = gamma

    def forward(self, logits, targets):
        bce = self.bce(logits.squeeze(), targets)

        prob = torch.sigmoid(logits.squeeze())
        pt = prob * targets + (1 - prob) * (1 - targets)

        loss = (1 - pt) ** self.gamma * bce
        return loss.mean()

device = "cuda" if torch.cuda.is_available() else "cpu"

model = Model().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
criterion = FocalLoss()


## 4. Model Training
This block contains the main training loop, where the model learns from the training data over several epochs. It tracks and prints the training loss for each epoch.

In [ ]:
for epoch in range(10):

    model.train()
    total_loss = 0

    for x, y in tqdm(train_loader):

        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()

        logits = model(x)
        loss = criterion(logits, y)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader):.4f}")


## 5. Model Evaluation
After training, the model's performance is evaluated on the unseen test dataset. The F1 score, a common metric for classification tasks, is calculated and printed.

In [ ]:
model.eval()

y_true = []
y_pred = []

with torch.no_grad():
    for x, y in test_loader:

        x = x.to(device)

        logits = model(x)
        probs = torch.sigmoid(logits).cpu().numpy()

        preds = (probs > 0.5).astype(int)

        y_true.extend(y.numpy())
        y_pred.extend(preds)

f1 = f1_score(y_true, y_pred)

print("FINAL F1 SCORE:", f1)


## 6. Confusion Matrix
This section generates and visualizes the confusion matrix to evaluate the model's performance in more detail, showing true positives, true negatives, false positives, and false negatives.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(6, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=["OK", "Defect"],
            yticklabels=["OK", "Defect"])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

## 7. Optional: Test Time Augmentation (TTA)
Test Time Augmentation (TTA) can improve prediction robustness by performing inference on augmented versions of the test image and averaging the results. Here's a helper function for TTA and a demonstration of its usage.

In [ ]:
def tta_predict(model, x):

    # x: single image tensor (1,3,224,224)

    preds = []

    with torch.no_grad():

        # original
        preds.append(torch.sigmoid(model(x)).item())

        # flip horizontally
        preds.append(torch.sigmoid(model(torch.flip(x, dims=[3]))).item())

        # flip vertically
        preds.append(torch.sigmoid(model(torch.flip(x, dims=[2]))).item())

    return np.mean(preds)

In [ ]:
# Demonstrate TTA for a sample image from the test set
sample_img, sample_label = test_ds[0]
sample_img = sample_img.unsqueeze(0).to(device) # Add batch dimension and move to device

tta_prediction = tta_predict(model, sample_img)
tta_result = "DEFECT" if tta_prediction > BEST_THRESHOLD else "GOOD"

print(f"Sample image actual label: {'DEFECT' if sample_label.item() == 1 else 'OK'}")
print(f"TTA Predicted Probability: {tta_prediction:.4f}")
print(f"TTA Prediction: {tta_result}")

## 8. Save and Download PyTorch Model
This section saves the trained PyTorch model's state dictionary and other relevant metadata to a `.pth` file, and then offers it for download.

In [ ]:
BEST_THRESHOLD = 0.5

torch.save({
    "model_state_dict": model.state_dict(),
    "threshold": BEST_THRESHOLD,
    "img_size": 224,
    "model_name": "tf_efficientnet_b0.ns_jft_in1k"
}, "casting_model.pth")

from google.colab import files
files.download("casting_model.pth")


## 9. Load PyTorch Model for ONNX Conversion
Before converting to ONNX, we load the saved PyTorch model back into memory and set it to evaluation mode. This ensures the model is ready for export.

In [ ]:
checkpoint = torch.load("casting_model.pth", map_location="cpu")

model = Model()
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

BEST_THRESHOLD = checkpoint["threshold"]
IMG_SIZE = checkpoint["img_size"]


## 10. Install ONNX Libraries
This cell ensures that the necessary ONNX and ONNX Runtime libraries are installed for model conversion and inference.

In [ ]:
pip install onnx onnxruntime onnxscript


## 11. Export to ONNX
Here, the PyTorch model is exported to the ONNX format. A dummy input is used to trace the model's computation graph during the export process.

In [ ]:
dummy_input = torch.randn(1,3,224,224)

torch.onnx.export(
    model,
    dummy_input,
    "casting_model.onnx",
    input_names=["input"],
    output_names=["output"],
    do_constant_folding=True,
    dynamic_axes=None,
    opset_version=13
)


## 12. Merge ONNX Model Files
If the ONNX model has external weights (i.e., weights stored in separate files), this step merges them into a single `.onnx` file for easier deployment.

In [ ]:
import onnx

# Load ONNX with external weights
model = onnx.load_model("/content/casting_model.onnx", load_external_data=True)

# Save as single file
onnx.save_model(
    model,
    "model_single.onnx",
    save_as_external_data=False
)

print("Merged successfully!")


## 13. Downgrade ONNX IR Version
This step addresses compatibility issues by downgrading the ONNX Intermediate Representation (IR) version of the model to a specified version (IR version 8 in this case).

In [ ]:
model = onnx.load("/content/model_single.onnx")
model.ir_version = 8  # Downgrade IR version
onnx.save(model, "your_model_ir8.onnx")


## 14. Verify ONNX IR Version
Finally, we load the modified ONNX model and print its IR version to confirm the downgrade was successful.

In [ ]:
model = onnx.load("/content/your_model_ir8.onnx")
print("IR version:", model.ir_version)


## 15. Run Inference with ONNX Runtime
This section demonstrates how to load the ONNX model using ONNX Runtime and perform inference on a sample image. It includes preprocessing steps to prepare the image for the model and post-processing to interpret the model's output.

In [ ]:
import onnxruntime as ort
import numpy as np
import cv2

# -------------------------------
# ✅ Your correct preprocessing
# -------------------------------
def preprocess(image_bytes):
    nparr = np.frombuffer(image_bytes, np.uint8)
    img = cv2.imdecode(nparr, cv2.IMREAD_GRAYSCALE)

    # grayscale → RGB (same as training)
    img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)

    # resize
    img = cv2.resize(img, (224, 224))

    # scale
    img = img.astype(np.float32) / 255.0

    # normalize (ImageNet)
    mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
    std  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

    img = (img - mean) / std

    # HWC → CHW
    img = np.transpose(img, (2, 0, 1))

    # add batch
    img = np.expand_dims(img, axis=0)

    return img


# -------------------------------
# ✅ Load ONNX model
# -------------------------------
session = ort.InferenceSession("/content/your_model_ir8.onnx") # Corrected filename

input_name = session.get_inputs()[0].name
output_name = session.get_outputs()[0].name

print("Input:", input_name)
print("Output:", output_name)


# -------------------------------
# ✅ Read image as bytes
# -------------------------------
with open("/content/drive/MyDrive/DL/casting_512x512/casting_512x512/ok_front/cast_ok_0_1018.jpeg", "rb") as f:
    image_bytes = f.read()

# preprocess using YOUR pipeline
x = preprocess(image_bytes)

print("Input shape:", x.shape)


# -------------------------------
# ✅ Run inference
# -------------------------------
pred = session.run([output_name], {input_name: x})[0]

# -------------------------------
# ✅ Convert logits → probability
# -------------------------------
logit = pred[0][0]

prob = 1 / (1 + np.exp(-logit))   # sigmoid

print("Logit:", logit)
print("Probability:", prob)

# -------------------------------
# ✅ Final classification
# -------------------------------
result = "DEFECT" if prob > 0.4 else "GOOD"

print("Prediction:", result)
